# 62. Ogilvie et al. 1995 — SWE on WIND / WIND 위성 SWE 구현

**Paper**: Ogilvie, K. W. et al., "SWE, A Comprehensive Plasma Instrument for the Wind Spacecraft", *Space Science Reviews* **71**, 55–77, 1995. DOI: [10.1007/BF00751326](https://doi.org/10.1007/BF00751326)

This notebook reproduces the key measurement principles of the Solar Wind Experiment (SWE) on the WIND spacecraft using synthetic plasma data. We implement four core analyses:

이 노트북은 합성 플라즈마 데이터를 사용해 WIND 위성 태양풍 실험기(SWE)의 핵심 측정 원리를 재현한다. 네 가지 분석을 구현한다.

1. **Faraday cup current** $I(V)$ as a function of solar-wind speed and temperature (Eq. 4.1, Figure 7).
2. **Maxwellian moment integration** to recover $n$, $\vec{u}$, $T$ from a velocity distribution function (Eq. 4.2).
3. **Electron strahl** angular distribution and field-aligned beaming (Section 3.5).
4. **Wind/SWE 30-year time series** showing the bulk speed, density and temperature variability across solar cycles (synthetic).

1. 패러데이 컵 전류 $I(V)$의 태양풍 속도/온도 의존성 (식 4.1, 그림 7).
2. 분포함수로부터 $n$, $\vec{u}$, $T$를 복원하는 맥스웰 모멘트 적분 (식 4.2).
3. 전자 strahl 각 분포와 자기력선 정렬 빔 (3.5절).
4. 30년치 WIND/SWE 시계열 시각화(합성).

All data are **synthetic** but use realistic 1 AU solar-wind parameters. Kernel: `study-with-ai`.

모든 데이터는 합성이지만 1 AU 태양풍의 실제 매개변수 범위를 따른다. 커널: `study-with-ai`.

In [ ]:
"""Imports and global plotting style.

Establish a reproducible random-number stream for synthetic data and configure
matplotlib defaults that match the publication style of Space Science Reviews.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, optimize

RNG = np.random.default_rng(seed=19951101)  # WIND launch date as seed

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "lines.linewidth": 1.6,
})

# Physical constants (SI).
K_B = 1.380649e-23  # Boltzmann constant [J/K].
M_P = 1.67262192e-27  # proton mass [kg].
M_E = 9.1093837e-31  # electron mass [kg].
Q_E = 1.602176634e-19  # elementary charge [C].
EV_TO_J = Q_E  # 1 eV in Joule.

## Part 1: Faraday Cup Current $I(V)$ / 패러데이 컵 전류 $I(V)$

The chopped current at the FC collector when the modulator gates a velocity window $[v_1, v_2]$ along the cup normal $\hat{n}$ (Eq. 4.1 of the notes):

변조기가 컵 법선 방향으로 $[v_1, v_2]$ 속도 창을 만들 때 콜렉터 전류:

$$
I(\hat{n}, v_1, v_2) = q\, A_\text{eff}(\theta) \int_{v_1}^{v_2} v_n F(v_n;\hat{n})\, dv_n,
$$

with $v_{1,2} = \sqrt{2qV_{1,2}/m}$ and the reduced 1-D Maxwellian $F_p(v_n) = n_p (m_p / 2\pi k_B T_p)^{1/2} \exp[-m_p (v_n - u_n)^2 / 2 k_B T_p]$.

Below we sweep modulator voltage from below the proton beam to above it and observe how $I(V)$ varies with bulk speed and temperature — the basis of SWE's single-spin mode.

변조 전압을 양성자 빔보다 낮은 값에서 높은 값까지 스위프하며 벌크 속도/온도에 따라 $I(V)$가 어떻게 변하는지 관찰한다 — SWE 단일 회전 모드의 기반.

In [ ]:
def reduced_maxwellian(v_n, u_n, w_p, n_p):
    """Compute the 1-D reduced Maxwellian along a cup normal.

    Args:
        v_n: Speed along cup normal [m/s], scalar or array.
        u_n: Bulk-flow component along cup normal [m/s].
        w_p: Proton thermal speed sqrt(2 k_B T_p / m_p) [m/s].
        n_p: Proton number density [m^-3].

    Returns:
        F_p in [s/m^4] (1-D number density per unit speed).
    """
    norm = n_p / (np.sqrt(np.pi) * w_p)
    return norm * np.exp(-((v_n - u_n) ** 2) / (w_p ** 2))


def fc_current(V_volts, dV_volts, u_n, w_p, n_p, A_eff_cm2=35.0, q=Q_E, m=M_P):
    """Integrate the FC chopped-current equation over a modulator window.

    Args:
        V_volts: Lower modulator voltage [V].
        dV_volts: Window width [V].
        u_n: Bulk-flow component along cup normal [m/s].
        w_p: Proton thermal speed [m/s].
        n_p: Proton density [m^-3].
        A_eff_cm2: Effective collecting area at theta=0 [cm^2].
        q: Particle charge [C].
        m: Particle mass [kg].

    Returns:
        Collector current in amperes.
    """
    A_eff = A_eff_cm2 * 1e-4  # cm^2 -> m^2.
    v_lo = np.sqrt(2 * q * V_volts / m)
    v_hi = np.sqrt(2 * q * (V_volts + dV_volts) / m)
    integrand = lambda v: v * reduced_maxwellian(v, u_n, w_p, n_p)
    val, _ = integrate.quad(integrand, v_lo, v_hi)
    return q * A_eff * val


# Sweep 64 logarithmic windows from 150 V to 8 kV (Table II).
V_levels = np.geomspace(150.0, 8000.0, 64)
dV_levels = 0.065 * V_levels  # narrow-window deltaE/E = 0.065.

# Reference solar wind: 400 km/s, T_p = 5e4 K, n_p = 5 cm^-3.
u_ref = 400e3
w_ref = np.sqrt(2 * K_B * 5e4 / M_P)
n_ref = 5e6  # 5 cm^-3 in m^-3.

I_curve = np.array([
    fc_current(V, dV, u_ref, w_ref, n_ref) for V, dV in zip(V_levels, dV_levels)
])

# Report the peak voltage and equivalent E/q.
i_peak = np.argmax(I_curve)
print(f"Peak modulator voltage: {V_levels[i_peak]:.0f} V")
print(f"Predicted from u: {0.5 * M_P * u_ref ** 2 / Q_E:.0f} V")
print(f"Peak current: {I_curve[i_peak] * 1e12:.1f} pA")

In [ ]:
"""Plot I(V) for a grid of (u, T) pairs and visualize the spectral peak shift."""
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

speeds_kms = [300, 400, 500, 700]
for u_kms in speeds_kms:
    I_arr = np.array([
        fc_current(V, dV, u_kms * 1e3, w_ref, n_ref) for V, dV in zip(V_levels, dV_levels)
    ])
    axes[0].semilogy(V_levels, np.maximum(I_arr * 1e12, 1e-3), label=f"u = {u_kms} km/s")
axes[0].set_xlabel("Modulator voltage V [V]")
axes[0].set_ylabel("Current [pA]")
axes[0].set_title("FC spectrum vs bulk speed / 벌크 속도별 FC 스펙트럼")
axes[0].legend()
axes[0].set_xlim(150, 4000)

temps_K = [2e4, 5e4, 1e5, 3e5]
for T_K in temps_K:
    w = np.sqrt(2 * K_B * T_K / M_P)
    I_arr = np.array([
        fc_current(V, dV, u_ref, w, n_ref) for V, dV in zip(V_levels, dV_levels)
    ])
    axes[1].plot(V_levels, I_arr * 1e12, label=f"T = {T_K:.0e} K")
axes[1].set_xlabel("Modulator voltage V [V]")
axes[1].set_ylabel("Current [pA]")
axes[1].set_title("FC spectrum vs temperature / 온도별 FC 스펙트럼")
axes[1].set_xlim(400, 1500)
axes[1].legend()

plt.tight_layout()
plt.show()

**Observation / 관찰**: As bulk speed $u$ increases the spectrum centroid moves to higher $V$ (since $V \propto u^2$). As temperature increases the peak broadens and lowers in amplitude — exactly the diagnostic SWE uses. A single Faraday-cup spectrum thus encodes $\{u, T, n\}$ simultaneously.

벌크 속도 $u$가 커지면 스펙트럼 중심이 더 높은 $V$로 이동($V \propto u^2$). 온도가 커지면 봉이 넓어지고 진폭이 낮아진다 — SWE가 사용하는 진단법 그대로다. 하나의 FC 스펙트럼이 $\{u, T, n\}$을 동시에 인코딩한다.

## Part 2: Twin-Peak Trace in a Single Spin / 단일 회전 쌍봉 곡선

Equation 4.7 of the notes shows that, when the modulator window is fixed just below the bulk-speed peak, the FC current as the spacecraft spins becomes a twin-peak function of azimuth $\theta$:

변조 창을 벌크 봉 직하에 고정하면, 위성이 자전할 때 azimuth $\theta$의 함수로서 전류가 쌍봉을 이룬다 (식 4.7).

$$
I(\theta) \propto \exp\!\left[-\frac{(v_M - u\cos\theta)^2}{w_p^2}\right] \cdot A_\text{eff}(\theta).
$$

We reproduce Figure 7 of the paper for five voltage windows scanning across the peak. Twin-peak separation $\to T$, plateau width vs window $\to u$, plateau height $\to n$.

다섯 개 변조 창에 대한 그림 7을 재현. 쌍봉 분리 $\to T$, 평탄대 위치 $\to u$, 평탄 높이 $\to n$.

In [ ]:
def aeff_response(theta_rad, theta_max=np.deg2rad(60)):
    """Approximate FC effective-area angular response from Figure 6c.

    Args:
        theta_rad: Incidence angle from cup normal [rad].
        theta_max: Geometric half-angle limit [rad], default ~60 deg.

    Returns:
        Effective area normalized to 35 cm^2 at theta=0.
    """
    val = 35.0 * np.cos(theta_rad) ** 2
    return np.where(np.abs(theta_rad) < theta_max, val, 0.0)


def fc_current_at_angle(theta_rad, V_volts, dV_volts, u, w_p, n_p):
    """FC current as a function of azimuth angle for a fixed modulator window."""
    u_n = u * np.cos(theta_rad)
    A_eff_cm2 = aeff_response(theta_rad)
    return fc_current(V_volts, dV_volts, u_n, w_p, n_p, A_eff_cm2=A_eff_cm2)


# Replicate Figure 7: u = 400 km/s, w = 40 km/s, n = 10 cm^-3.
u_fig7 = 400e3
w_fig7 = 40e3
n_fig7 = 10e6
T_fig7 = 0.5 * M_P * w_fig7 ** 2 / K_B
print(f"Equivalent T_p = {T_fig7:.2e} K")

# Voltage windows of Figure 7 (energies in eV correspond to mid-window).
windows_eV = [(309, 329), (329, 350), (350, 373), (373, 397), (397, 423)]
windows_V = [
    (0.5 * M_P * (lo * 1e3) ** 2 / Q_E, 0.5 * M_P * (hi * 1e3) ** 2 / Q_E)
    for lo, hi in windows_eV
]

thetas_deg = np.linspace(-50, 50, 201)
thetas_rad = np.deg2rad(thetas_deg)

fig, axes = plt.subplots(5, 1, figsize=(8, 11), sharex=True)
for ax, (lo, hi), (V_lo, V_hi) in zip(axes, windows_eV, windows_V):
    dV = V_hi - V_lo
    I_theta = np.array([
        fc_current_at_angle(t, V_lo, dV, u_fig7, w_fig7, n_fig7) for t in thetas_rad
    ])
    ax.plot(thetas_deg, I_theta * 1e12, color="C0")
    ax.set_ylabel("I [pA]")
    ax.set_title(f"Window {lo}–{hi} km/s", fontsize=10)
    ax.axhline(0, color="gray", lw=0.5)
axes[-1].set_xlabel("Azimuth angle from Sun [deg]")
fig.suptitle("Single-spin twin-peak traces (cf. Ogilvie 1995, Figure 7)", fontsize=12)
plt.tight_layout()
plt.show()

## Part 3: Maxwellian Moment Integration / 맥스웰 모멘트 적분

From a 3-D VDF $f(\vec{v})$ the lowest moments give density, bulk velocity, and temperature:

3차원 분포함수 $f(\vec{v})$의 저차 모멘트로 밀도·벌크 속도·온도를 추출한다.

$$
n = \int f\, d^3 v, \qquad \vec{u} = \frac{1}{n} \int \vec{v} f\, d^3 v, \qquad T = \frac{m}{3 n k_B} \int |\vec{v} - \vec{u}|^2 f\, d^3 v.
$$

Below we generate a synthetic drifting Maxwellian VDF on a Cartesian velocity grid, integrate to recover $\{n, \vec{u}, T\}$, and verify accuracy.

이래에서는 합성 드리프트 맥스웰 분포를 카르테시안 속도 격자 위에 만들고, 적분해 $\{n, \vec{u}, T\}$를 복원한 뒤 정확도를 검증한다.

In [ ]:
def drifting_maxwellian_3d(vx, vy, vz, n_p, u_vec, T_K):
    """Build a 3-D drifting Maxwellian VDF.

    Args:
        vx, vy, vz: Velocity-grid coordinates [m/s] (broadcastable).
        n_p: Number density [m^-3].
        u_vec: Bulk velocity [m/s] (length-3 array).
        T_K: Temperature [K].

    Returns:
        f_p [s^3/m^6].
    """
    w2 = 2 * K_B * T_K / M_P
    norm = n_p * (1 / (np.pi * w2)) ** 1.5
    arg = ((vx - u_vec[0]) ** 2 + (vy - u_vec[1]) ** 2 + (vz - u_vec[2]) ** 2) / w2
    return norm * np.exp(-arg)


def vdf_moments(vx, vy, vz, f, mass=M_P):
    """Integrate a VDF on a regular grid to obtain n, u, T.

    Args:
        vx, vy, vz: 1-D coordinate arrays [m/s].
        f: 3-D VDF on the meshgrid produced from (vx, vy, vz).
        mass: Particle mass [kg].

    Returns:
        Tuple (n_p [m^-3], u_vec [m/s], T_K [K]).
    """
    dvx, dvy, dvz = np.diff(vx)[0], np.diff(vy)[0], np.diff(vz)[0]
    dV = dvx * dvy * dvz
    Vx, Vy, Vz = np.meshgrid(vx, vy, vz, indexing="ij")
    n = float(np.sum(f) * dV)
    ux = float(np.sum(Vx * f) * dV / n)
    uy = float(np.sum(Vy * f) * dV / n)
    uz = float(np.sum(Vz * f) * dV / n)
    sq = (Vx - ux) ** 2 + (Vy - uy) ** 2 + (Vz - uz) ** 2
    T = float(mass * np.sum(sq * f) * dV / (3 * n * K_B))
    return n, np.array([ux, uy, uz]), T


# True parameters.
true_n = 7.0e6
true_u = np.array([-420e3, 25e3, -10e3])
true_T = 8e4

# Velocity grid centered on u with +-5 thermal speeds.
w_true = np.sqrt(2 * K_B * true_T / M_P)
v_axis = np.linspace(-5 * w_true, 5 * w_true, 81)
vx = v_axis + true_u[0]
vy = v_axis + true_u[1]
vz = v_axis + true_u[2]
Vx, Vy, Vz = np.meshgrid(vx, vy, vz, indexing="ij")
f_grid = drifting_maxwellian_3d(Vx, Vy, Vz, true_n, true_u, true_T)

rec_n, rec_u, rec_T = vdf_moments(vx, vy, vz, f_grid)
print(f"Recovered n = {rec_n / 1e6:.3f} cc^-1   (true {true_n / 1e6:.3f})")
print(f"Recovered u = {rec_u / 1e3} km/s    (true {true_u / 1e3})")
print(f"Recovered T = {rec_T:.2e} K       (true {true_T:.2e})")
print(f"Errors: n {(rec_n - true_n) / true_n * 100:.2f}%, T {(rec_T - true_T) / true_T * 100:.2f}%")

In [ ]:
"""Visualize 2-D slices of the recovered VDF in (vx, vy) and (vx, vz) planes."""
f_xy = f_grid.sum(axis=2) * np.diff(vz)[0]  # marginalize vz.
f_xz = f_grid.sum(axis=1) * np.diff(vy)[0]  # marginalize vy.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, plane, xy_data, xy_labels in (
    (axes[0], "xy", f_xy, ("v_x [km/s]", "v_y [km/s]", vy / 1e3)),
    (axes[1], "xz", f_xz, ("v_x [km/s]", "v_z [km/s]", vz / 1e3)),
):
    pcm = ax.pcolormesh(
        vx / 1e3, xy_labels[2], xy_data.T, shading="auto", cmap="viridis"
    )
    ax.scatter([rec_u[0] / 1e3], [rec_u[1 if plane == "xy" else 2] / 1e3],
               c="red", s=60, marker="x", label="recovered u")
    ax.set_xlabel(xy_labels[0])
    ax.set_ylabel(xy_labels[1])
    ax.set_title(f"f marginalized over v_{'z' if plane == 'xy' else 'y'}")
    ax.legend()
    fig.colorbar(pcm, ax=ax, label="reduced f")
plt.tight_layout()
plt.show()

**Verification / 검증**: The recovered moments match the inputs to better than 0.1%, confirming that a Cartesian grid spanning roughly $\pm 5 w_p$ around the bulk velocity is sufficient. SWE's actual moment routine uses similar quadrature on a polar (energy-angle) grid produced by the FC modulator scans.

복원된 모멘트가 입력과 0.1% 이내로 일치하므로 벌크 속도 주위 $\pm 5 w_p$ 카르테시안 격자로 충분함을 확인했다. SWE의 실제 모멘트 루틴은 FC 변조 스캔이 만드는 극좌표(에너지-각) 격자에서 동일한 적분을 수행한다.

## Part 4: Electron Strahl Angular Distribution / 전자 strahl 각 분포

Solar-wind electrons split into three populations: a thermal **core** (Maxwellian, $T_c \sim 10^5$ K), a hot **halo** (kappa-like), and a narrow field-aligned **strahl** beam at $E > 40$ eV. SWE's strahl detector resolves the strahl with $\sim 5°$ angular resolution (Section 3.5).

태양풍 전자는 세 성분으로 나뉜다 — 열적 코어(맥스웰, $T_c \sim 10^5$ K), 뜨거운 할로(카파 형태), 자기력선 정렬 좁은 strahl 빔($E > 40$ eV). SWE strahl 검출기는 약 5° 각 분해능으로 strahl을 분해한다.

We construct a synthetic pitch-angle distribution and compute the **first-order anisotropy** $A_1 = \int f \cos\alpha\, d\Omega / \int f\, d\Omega$, a proxy for the heat-flux direction.

합성 pitch-angle 분포를 만들고 **1차 비등방성** $A_1 = \int f \cos\alpha\, d\Omega / \int f\, d\Omega$을 계산하며, 이는 열속 방향의 대리 지표다.

In [ ]:
def core_halo_strahl_pad(alpha_rad, n_core=8e6, n_halo=4e5, n_strahl=2e5,
                          strahl_width_deg=8.0):
    """Synthetic electron pitch-angle distribution at fixed energy E ~ 200 eV.

    Args:
        alpha_rad: Pitch angle from B-field [rad], 0 = field-aligned outward.
        n_core: Density of isotropic core component.
        n_halo: Density of broad halo component.
        n_strahl: Density of narrow strahl beam.
        strahl_width_deg: Half-width of strahl Gaussian in pitch angle [deg].

    Returns:
        Phase-space density value (relative units).
    """
    sigma = np.deg2rad(strahl_width_deg)
    core = n_core / (4 * np.pi)  # isotropic.
    halo = (n_halo / (4 * np.pi)) * (1 + 0.05 * np.cos(alpha_rad))  # mild anisotropy.
    strahl = (n_strahl / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-(alpha_rad ** 2) / (2 * sigma ** 2))
    return core + halo + strahl


alpha_deg = np.linspace(-90, 270, 721)  # 0.5-deg sampling.
alpha_rad = np.deg2rad(alpha_deg)

# Pitch angle alpha is measured from B; wrap to range [0, 180] for plot.
alpha_phys = np.where(alpha_deg < 0, -alpha_deg, alpha_deg)
alpha_phys = np.where(alpha_phys > 180, 360 - alpha_phys, alpha_phys)

f_pad = core_halo_strahl_pad(np.deg2rad(alpha_phys))

# First-order anisotropy A1.
alpha_full = np.linspace(0, np.pi, 361)
f_full = core_halo_strahl_pad(alpha_full)
weight = np.sin(alpha_full)
A1 = np.trapz(f_full * np.cos(alpha_full) * weight, alpha_full) / np.trapz(f_full * weight, alpha_full)
print(f"First-order anisotropy A1 = {A1:.3f}")
print("Positive A1 indicates net flow along +B (outward heat flux).")

In [ ]:
"""Polar plot of the strahl + halo + core pitch-angle distribution."""
fig = plt.figure(figsize=(13, 6))

ax1 = fig.add_subplot(1, 2, 1, projection="polar")
theta = np.linspace(0, 2 * np.pi, 721)
f_polar = core_halo_strahl_pad(np.where(theta <= np.pi, theta, 2 * np.pi - theta))
ax1.plot(theta, f_polar, color="C0")
ax1.fill(theta, f_polar, alpha=0.2)
ax1.set_theta_zero_location("E")
ax1.set_title("Electron PAD (core + halo + strahl)\n전자 PAD")
ax1.set_rscale("log")
ax1.set_rlim(1e4, 5e6)

ax2 = fig.add_subplot(1, 2, 2)
alpha_axis = np.linspace(0, 180, 361)
f_lin = core_halo_strahl_pad(np.deg2rad(alpha_axis))
ax2.semilogy(alpha_axis, f_lin, color="C0")
ax2.axvspan(0, 16, color="red", alpha=0.1, label="~strahl FOV")
ax2.set_xlabel("Pitch angle alpha [deg]")
ax2.set_ylabel("phase-space density [arb]")
ax2.set_title("Strahl peak at alpha=0 / strahl 봉 (alpha=0)")
ax2.legend()
ax2.set_xlim(0, 180)

plt.tight_layout()
plt.show()

**Strahl signature / strahl 특징**: A clear narrow peak at $\alpha = 0$ rides on top of the broad halo. SWE's strahl detector with $\Delta E/E = 0.03$ and $\sim 5°$ anodes resolves this peak; the 7° FOV of VEIS would partially smear it. The positive $A_1$ encodes the field-aligned heat flux — an essential observable for foreshock and CME-driven structures.

$\alpha = 0$의 좁은 봉이 넓은 halo 위에 얹혀 있다. SWE strahl 검출기는 ΔE/E=0.03, ~5° anode로 이 봉을 분해한다. VEIS의 7° FOV는 부분적으로 흐려놓는다. 양의 $A_1$이 자기력선 정렬 열속을 인코딩 — 전방 충격파역과 CME 유도 구조 연구의 필수 관측량.

## Part 5: Wind/SWE 30-Year Time Series (Synthetic) / WIND/SWE 30년 시계열 (합성)

WIND has been operating since November 1994 — over 30 years. Below we produce a synthetic time series for $V_\text{sw}$, $n_p$ and $T_p$ at hourly cadence from 1995 through 2025 with realistic features:

WIND는 1994년 11월부터 30년 넘게 운영 중이다. 1995-2025년의 $V_\text{sw}$, $n_p$, $T_p$를 시간 분해능으로 다음 특성을 포함해 합성한다.

- 11-year solar cycle modulation (cycles 23, 24, 25)
- 27-day Carrington-rotation periodicity
- log-normal CME / high-speed-stream excursions
- correlated $V$–$T$ via empirical Lopez (1987) relation $T_p \approx (0.031 V_\text{sw} - 5.1)^2$ K

11년 태양 주기 변조(주기 23, 24, 25), 27일 캐링턴 자전 주기성, 대수정규 CME/고속류 변동, 그리고 Lopez(1987)의 경험적 $V$-$T$ 관계.

In [ ]:
def synthesize_wind_series(start_year=1995.0, end_year=2025.0, dt_hours=1.0):
    """Synthesize WIND/SWE-like time series of V_sw, n_p, T_p.

    Args:
        start_year: Float year for series start.
        end_year: Float year for series end.
        dt_hours: Sample cadence [hours].

    Returns:
        Tuple of arrays (year, V_sw_kms, n_p_cc, T_p_K).
    """
    n_pts = int((end_year - start_year) * 365.25 * 24 / dt_hours)
    year = np.linspace(start_year, end_year, n_pts)

    # Solar-cycle phase: minima ~ 1996.5, 2008.9, 2019.9.
    cycle_phase = 2 * np.pi * (year - 1996.5) / 11.0
    cycle_amp = 0.5 * (1 + np.cos(cycle_phase))  # 0..1, max at solar max.

    # 27-day Carrington-stream component.
    carrington = 0.5 * (1 + np.sin(2 * np.pi * year * 365.25 / 27.0))

    # Quiet baseline + cycle-modulated speed.
    V_base = 380 + 60 * cycle_amp + 80 * carrington
    V_noise = RNG.normal(0, 30, n_pts)

    # CME-like log-normal excursions (more during solar max).
    cme_prob = 0.0005 + 0.002 * cycle_amp
    cme_mask = RNG.random(n_pts) < cme_prob
    cme_boost = np.zeros(n_pts)
    cme_boost[cme_mask] = RNG.lognormal(mean=4.5, sigma=0.4, size=cme_mask.sum())

    V_sw = V_base + V_noise + cme_boost
    V_sw = np.clip(V_sw, 280, 1100)

    # Density: anti-correlated with speed (slower wind is denser).
    n_p = 12 * (400 / V_sw) ** 1.5 * RNG.lognormal(0, 0.3, n_pts)
    n_p = np.clip(n_p, 0.5, 60)

    # Temperature via Lopez (1987) relation.
    T_p = np.maximum((0.031 * V_sw - 5.1) ** 2 * 1e3, 1e4)
    T_p *= RNG.lognormal(0, 0.2, n_pts)

    return year, V_sw, n_p, T_p


year, V_sw, n_p_cc, T_p = synthesize_wind_series()
print(f"Synthesized {len(year):,} hourly samples.")
print(f"Mean V_sw = {V_sw.mean():.0f} km/s   (median {np.median(V_sw):.0f})")
print(f"Mean n_p = {n_p_cc.mean():.1f} cc^-1")
print(f"Mean T_p = {T_p.mean():.1e} K")

In [ ]:
"""Plot the 30-year SWE time series with monthly smoothing for clarity."""
samples_per_month = int(30 * 24)
kernel = np.ones(samples_per_month) / samples_per_month
V_smooth = np.convolve(V_sw, kernel, mode="same")
n_smooth = np.convolve(n_p_cc, kernel, mode="same")
T_smooth = np.convolve(T_p, kernel, mode="same")

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(year, V_sw, color="C0", lw=0.3, alpha=0.4, label="hourly")
axes[0].plot(year, V_smooth, color="C3", lw=1.5, label="30-day mean")
axes[0].set_ylabel("V_sw [km/s]")
axes[0].legend(loc="upper right")
axes[0].set_title("WIND/SWE 30-year synthetic time series")

axes[1].semilogy(year, n_p_cc, color="C0", lw=0.3, alpha=0.4)
axes[1].semilogy(year, n_smooth, color="C3", lw=1.5)
axes[1].set_ylabel("n_p [cm^-3]")

axes[2].semilogy(year, T_p, color="C0", lw=0.3, alpha=0.4)
axes[2].semilogy(year, T_smooth, color="C3", lw=1.5)
axes[2].set_ylabel("T_p [K]")
axes[2].set_xlabel("Year")

for ax in axes:
    for sc_min in (1996.5, 2008.9, 2019.9):
        ax.axvline(sc_min, color="gray", ls=":", lw=0.8)
    for sc_max in (2001.0, 2014.0, 2024.5):
        ax.axvline(sc_max, color="orange", ls=":", lw=0.8)

plt.tight_layout()
plt.show()

In [ ]:
"""Joint distribution: V_sw vs T_p (recover Lopez 1987 trend)."""
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hexbin(V_sw, T_p, gridsize=60, yscale="log", bins="log", cmap="viridis")
v_grid = np.linspace(300, 800, 100)
T_lopez = (0.031 * v_grid - 5.1) ** 2 * 1e3
axes[0].plot(v_grid, T_lopez, "r-", lw=2, label="Lopez 1987")
axes[0].set_xlabel("V_sw [km/s]")
axes[0].set_ylabel("T_p [K]")
axes[0].set_title("V-T correlation / V-T 상관")
axes[0].legend()

axes[1].hexbin(V_sw, n_p_cc, gridsize=60, yscale="log", bins="log", cmap="plasma")
axes[1].set_xlabel("V_sw [km/s]")
axes[1].set_ylabel("n_p [cm^-3]")
axes[1].set_title("V-n anti-correlation / V-n 반비례")

plt.tight_layout()
plt.show()

**Long-term observations / 장기 관측 결과**:

1. **Solar-cycle modulation** — average $V_\text{sw}$ is ~ 50 km/s higher near solar maxima 2001/2014/2024 than at minima 1996/2008/2019, consistent with HSS/CME activity tracking sunspot number.
2. **Lopez relation visible** — the V-T scatter follows $T_p \propto (0.031 V - 5.1)^2$ K (red curve), the empirical relation that has been verified across multiple solar cycles by SWE.
3. **V-n anti-correlation** — the slowest wind is the densest (slow streamer-belt origin) and the fastest wind is the most rarefied (coronal-hole origin) — fundamental observation that SWE established at unprecedented duration.

1. **태양 주기 변조** — 평균 $V_\text{sw}$는 극대기(2001/2014/2024) 부근이 극소기(1996/2008/2019)보다 약 50 km/s 높아 흑점수에 따라가는 HSS/CME 활동을 반영.
2. **Lopez 관계** — V-T 산포가 $T_p \propto (0.031 V - 5.1)^2$ K(빨간 곡선)을 따른다. SWE가 여러 태양 주기에 걸쳐 검증한 경험식.
3. **V-n 반비례** — 가장 느린 바람이 가장 조밀하고(streamer-belt 기원), 가장 빠른 바람이 가장 희박하다(코로나 홀 기원) — SWE가 전례 없는 기간 동안 확립한 기본 관측.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Faraday cup chopped current | 200 Hz lock-in modulator, FC1/FC2 | ACE/SWEPAM, DSCOVR/PlasMag, PSP/SPC |
| Reduced VDF integration $I = q A_\text{eff} \int v_n F\, dv$ | Eq. 4.1 (manuscript) | Same algebra used in all FC pipelines today |
| 3-D moment integration $n, \vec u, T$ | VEIS 6×16×6 grid | Solar Orbiter/SWA-PAS, MMS/FPI top-hat ESAs |
| Strahl detector (toroidal ESA) | 131° toroidal, ΔE/E=0.03, 5° anode | STEREO/IMPACT-SWEA, PSP/SPAN-E |
| Single-spin twin-peak inversion | 3-s spin gives $\{n, u, T\}$ | DSCOVR FC operational mode |
| V-T Lopez relation | Validated across cycles 22-25 | Used in solar-wind classification (Xu & Borovsky 2015) |

**Key insight / 핵심 통찰**: SWE's three-sensor architecture (FC + VEIS + strahl) under one DPU established the template for modern integrated solar-wind plasma instruments. The 30-year continuity at L1 makes WIND/SWE the gold-standard reference for any new heliospheric mission.

SWE의 세 센서 통합 구조(FC + VEIS + strahl)는 현대 통합 태양풍 플라즈마 측정기의 표준 템플릿을 확립했다. L1 30년 연속 운영으로 WIND/SWE는 모든 신규 헬리오스피어 미션의 황금 기준이다.